# Example 5: Computing p-values with CV-Aware Pretrained Lasso

This notebook demonstrates how to use `PPL_SI_CV` and `PPL_SI_CV_randj` to compute
selective p-values after data-driven hyperparameter selection via cross-validation.

The CV-aware inference conditions on the full CV selection event:

$$\mathcal{Z}_{\rm CV} = \mathcal{Z}_1 \cap \mathcal{Z}_2 \cap \mathcal{Z}_3$$

where $\mathcal{Z}_2$ encodes the Step 1 CV winner ($\lambda_{\rm sh}$) and $\mathcal{Z}_3$
encodes the Step 2 CV winner $(\rho, \lambda_K)$.

In [1]:
import numpy as np
import sys
sys.path.append('..')

from ppl_si import generate_synthetic_data, PPL_SI_CV, PPL_SI_CV_randj

## 1. Generate Synthetic Data

In [2]:
np.random.seed(42)

p = 300
num_sh = 10
num_inv = 5
K = 5
n_list = [100, 100, 100, 100, 100]
true_beta_sh = 0.3

X_list, Y_list, true_betaK, Sigma_list = generate_synthetic_data(
    p=p,
    num_sh=num_sh,
    num_inv=num_inv,
    K=K,
    n_list=n_list,
    true_beta_sh=true_beta_sh,
    itc=0.5,
    Gamma=0.05
)

print(f"Number of groups: {len(X_list)}")
print(f"Feature dimension: {p}")
print(f"Target sample size: {n_list[-1]}")

Number of groups: 5
Feature dimension: 300
Target sample size: 100


## 2. Set CV Hyperparameter Grids

In [ ]:
Lambda = [60, 90, 120]
Lambda_tilde = [
    (0.1, 5),
    (0.1, 10),
    (0.3, 5),
    (0.3, 10),
]

n_folds = 5
z_min = -20
z_max = 20
num_segments = 24

print(f"Lambda grid (lambda_sh):  {Lambda}")
print(f"Lambda_tilde grid (rho, lambda_K):")
for Phi in Lambda_tilde:
    print(f"  rho={Phi[0]}, lambda_K={Phi[1]}")
print(f"n_folds: {n_folds}")
print(f"num_segments: {num_segments}")

## 3. Compute p-values for All Selected Features

In [ ]:
p_values = PPL_SI_CV(
    X_list=X_list,
    Y_list=Y_list,
    Lambda=Lambda,
    Lambda_tilde=Lambda_tilde,
    Sigma_list=Sigma_list,
    n_folds=n_folds,
    z_min=z_min,
    z_max=z_max,
    num_segments=num_segments,
    seed=42
)

if p_values is not None:
    print(f"Number of selected features: {len(p_values)}")
    print("\nFeature index and p-values:")
    for j, p_val in p_values:
        print(f"Feature {j}: true_betaK[{j}] = {true_betaK[j]:.4f}, p-value = {p_val:.4f}")
else:
    print("No features selected")

## 4. Compute p-value for a Random Selected Feature

In [ ]:
j, p_value_rand = PPL_SI_CV_randj(
    X_list=X_list,
    Y_list=Y_list,
    Lambda=Lambda,
    Lambda_tilde=Lambda_tilde,
    Sigma_list=Sigma_list,
    n_folds=n_folds,
    z_min=z_min,
    z_max=z_max,
    num_segments=num_segments,
    seed=42
)

if p_value_rand is not None:
    print(f"Random feature: {j} - true_betaK[{j}] = {true_betaK[j]:.4f}, p-value = {p_value_rand:.4f}")
else:
    print("No features selected")

## 5. Analysis

In [ ]:
if p_values is not None:
    alpha = 0.05
    significant_features = [(j, p) for j, p in p_values if p <= alpha]

    print(f"Significance level: {alpha}")
    print(f"Number of selected features: {len(p_values)}")
    print(f"Number of significant features: {len(significant_features)}")

    if len(significant_features) > 0:
        print("\nSignificant features:")
        for j, p_val in significant_features:
            print(f"Feature {j}: true_betaK[{j}] = {true_betaK[j]:.4f}, p-value = {p_val:.4f}")